<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 03 · Model Evaluation

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
Chapter 5 is about comparison. The point is not to chase a fancy score, but
to make the evaluation path explicit and repeatable.

This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

module_path = Path('code') / 'baseline_models.py'
spec = importlib.util.spec_from_file_location('baseline_models', module_path)
assert spec is not None and spec.loader is not None
baseline_models = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline_models)

print(f'Loaded {module_path.resolve()}')


This cell continues the worked example for the current section.


In [ ]:
rng = np.random.default_rng(seed=11)
reg_frame = pd.DataFrame(
    {
        'feature': rng.normal(size=200),
    }
)
reg_frame['target'] = 3.0 * reg_frame['feature'] + rng.normal(scale=0.5, size=200)

rmse = baseline_models.baseline_linear(reg_frame, 'target')
print('baseline linear RMSE =', round(rmse, 3))


This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


In [ ]:
X, y = make_classification(
    n_samples=300,
    n_features=6,
    n_informative=3,
    n_redundant=1,
    weights=[0.7, 0.3],
    random_state=7,
)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=7,
    stratify=y,
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=7,
    stratify=y_temp,
)

models = {
    'logistic_regression': LogisticRegression(max_iter=1000),
    'decision_tree': DecisionTreeClassifier(max_depth=4, random_state=7),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    for split_name, X_split, y_split in [
        ('validation', X_valid, y_valid),
        ('test', X_test, y_test),
    ]:
        pred = model.predict(X_split)
        rows.append(
            {
                'model': name,
                'split': split_name,
                'accuracy': accuracy_score(y_split, pred),
                'precision': precision_score(y_split, pred, zero_division=0),
                'recall': recall_score(y_split, pred, zero_division=0),
            }
        )

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))


This cell continues the worked example for the current section.


In [ ]:
tree = models['decision_tree']
preds = tree.predict(X_test)
print(confusion_matrix(y_test, preds))


### Where We Are Heading Next
Chapter 6 shifts from evaluation to the feature choices that shape model
behavior before any metric is calculated.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
